### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\jeewa\AppData\Local\Temp\ipykernel_1444\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\jeewa\Desktop\RAG-project01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pathlib import Path

### Read all the PDFs inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: GOAT-Bench 2024.pdf
  ✓ Loaded 16 pages

Processing: IVLN (iterative vision language navigation) 2023.pdf
  ✓ Loaded 14 pages

Processing: LH-VLN 2024.pdf
  ✓ Loaded 17 pages

Total documents loaded: 47


### Text Splitting Parameters

`chunk_size` controls the maximum size of each chunk.

`chunk_overlap` controls how much text is repeated between neighboring chunks so context is not lost.

`length_function` tells the splitter how to measure size. Here, `len` means it counts characters.

`separators` defines the order of text boundaries the splitter tries when breaking content:
- `\n\n` for paragraph breaks
- `\n` for line breaks
- ` ` for spaces
- `` for any remaining text

Together, these four settings help create chunks that are small, readable, and good for retrieval.

In [6]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [7]:
chunks = split_documents(all_pdf_documents)
chunks

Split 47 documents into 268 chunks

Example chunk:
Content: GOAT-Bench: A Benchmark for Multi-Modal Lifelong Navigation
Mukul Khanna1∗ Ram Ramrakhya1∗ Gunjan Chhablani1 Sriram Yenamandra1 Theophile Gervet2
Matthew Chang3 Zsolt Kira1 Devendra Singh Chaplot4 Dhr...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-11T00:53:03+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-11T00:53:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\GOAT-Bench 2024.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'GOAT-Bench 2024.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-11T00:53:03+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-11T00:53:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\GOAT-Bench 2024.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'GOAT-Bench 2024.pdf', 'file_type': 'pdf'}, page_content='GOAT-Bench: A Benchmark for Multi-Modal Lifelong Navigation\nMukul Khanna1∗ Ram Ramrakhya1∗ Gunjan Chhablani1 Sriram Yenamandra1 Theophile Gervet2\nMatthew Chang3 Zsolt Kira1 Devendra Singh Chaplot4 Dhruv Batra1 Roozbeh Mottaghi5\n1Georgia Institute of Technology 2Carnegie Mellon University\n3University of Illinois Urbana-Champaign 4Mistral AI 5University of Washington\nmukulkhanna.github.io/goat-bench\nFigure 1. We study the Go to Any Thing (GOAT) task, which involves

### Embedding and VectorStoreDB 

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class Embedding_manager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()


##initialize the embedding manager
embedding_manager = Embedding_manager()
embedding_manager



Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5353.59it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\jeewa\AppData\Local\Temp\ipykernel_1444\327656720.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [10]:
### vectorStore  # Section title for this notebook part

class VectorStore:  # Define a class to manage ChromaDB operations
    """Manages document embeddings in a ChromaDB vector store"""  # Class description

    def __init__(  # Constructor: runs automatically when you create VectorStore(...)
        self,  # Reference to the current object instance
        collection_name: str = "pdf_documents",  # Default collection name in ChromaDB
        persist_directory: str = "../data/vector_store"  # Folder path to store DB files
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name  # Save collection name to object state
        self.persist_directory = persist_directory  # Save persistence path to object state
        self.client = None  # Placeholder for ChromaDB client (set in _initialize_store)
        self.collection = None  # Placeholder for collection object (set in _initialize_store)
        self._initialize_store()  # Immediately initialize DB client and collection

    def _initialize_store(self):  # Method to set up ChromaDB for this VectorStore instance
        """Initialize ChromaDB client and collection"""  # Short description of what this method does
        try:  # Start a protected block to catch initialization errors
            # Create the persistence folder if it doesn't already exist
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create a persistent ChromaDB client pointing to that folder
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get an existing collection by name, or create it if it doesn't exist
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,  # Collection name configured in __init__
                metadata={"description": "PDF document embeddings for RAG"}  # Optional collection metadata
            )

            # Log successful initialization and the active collection name
            print(f"Vector store initialized. Collection: {self.collection_name}")

            # Log how many documents are already stored in this collection
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:  # Catch any error during folder/client/collection setup
            print(f"Error initializing vector store: {e}")  # Print readable error message
            raise  # Re-raise the exception so caller can handle it

    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding (convert numpy row to plain list)
            if isinstance(embedding, np.ndarray):
                embeddings_list.append(embedding.tolist())
            else:
                embeddings_list.append(list(embedding))

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 268


In [11]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-11T00:53:03+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-11T00:53:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\GOAT-Bench 2024.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'GOAT-Bench 2024.pdf', 'file_type': 'pdf'}, page_content='GOAT-Bench: A Benchmark for Multi-Modal Lifelong Navigation\nMukul Khanna1∗ Ram Ramrakhya1∗ Gunjan Chhablani1 Sriram Yenamandra1 Theophile Gervet2\nMatthew Chang3 Zsolt Kira1 Devendra Singh Chaplot4 Dhruv Batra1 Roozbeh Mottaghi5\n1Georgia Institute of Technology 2Carnegie Mellon University\n3University of Illinois Urbana-Champaign 4Mistral AI 5University of Washington\nmukulkhanna.github.io/goat-bench\nFigure 1. We study the Go to Any Thing (GOAT) task, which involves

In [12]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]  
texts

['GOAT-Bench: A Benchmark for Multi-Modal Lifelong Navigation\nMukul Khanna1∗ Ram Ramrakhya1∗ Gunjan Chhablani1 Sriram Yenamandra1 Theophile Gervet2\nMatthew Chang3 Zsolt Kira1 Devendra Singh Chaplot4 Dhruv Batra1 Roozbeh Mottaghi5\n1Georgia Institute of Technology 2Carnegie Mellon University\n3University of Illinois Urbana-Champaign 4Mistral AI 5University of Washington\nmukulkhanna.github.io/goat-bench\nFigure 1. We study the Go to Any Thing (GOAT) task, which involves agents navigating to a sequence of open vocabulary goals specified\nthrough any of the three modalities – category name, a language description, or an image. We propose GOAT-Bench, a benchmark for the\nGOAT task, where we evaluate modular and monolithic, explicit and implicit map-based navigation approaches. In the above example, we\ntask the agent with sequentially navigating to 1) a recliner chair (from a closed set of k categories), 2) the oven shown in the picture, 3) “the',
 'task the agent with sequentially navig

In [13]:
##generate the embedidngs

embeddings = embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 268 texts...


Batches: 100%|██████████| 9/9 [00:03<00:00,  2.65it/s]


Generated embeddings with shape: (268, 384)
Adding 268 documents to vector store...
Successfully added 268 documents to vector store
Total documents in collection: 536


## Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: Embedding_manager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager




    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score to keep a result.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Create embedding for the user query (expects first vector from batch output).
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # Query ChromaDB using the query embedding.
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs: List[Dict[str, Any]] = []

            # Chroma returns nested lists: one list per query; we use the first query's results.
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity for easier threshold filtering.
                    # For cosine distance, higher similarity is better.
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            # Return empty list to keep app flow safe even if retrieval fails.
            print(f"Error during retrieval: {e}")
            return []
        

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [17]:
rag_retriever

In [40]:
rag_retriever.retrieve("Multi-Granularity Dynamic Memory Model")

Retrieving documents for query: 'Multi-Granularity Dynamic Memory Model'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 121.16it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_276520ac_200',
  'content': 'Avg Pool\nTake the book in living \nroom to the desk in office, \nthen take the clock in the \noffice to the kitchen \ncounter in kitchen.\nHistory\nObservation\nVit\nVision Encoder\nCoT Feedback\nSubtask finished \n& Few steps\nLLM\n(Vicuna7B)\n𝒔𝟏 𝒔𝟐 𝒔𝟑𝟎\nMulti-\nview\nFusion\n𝒉𝟎 𝒉𝟏 𝒉𝟐 𝒉𝒏\n𝒄𝟎 𝒄𝟏 𝒄𝟐 𝒄𝒏\n𝒉𝟑\n𝒄𝟑\n…\n…\nShort Term Memory\nAvg Pool\n𝐌𝐢𝐧(− \u0dcd\n𝒊\n𝒄𝒊 𝐥𝐨𝐠 𝒄𝒊 )\nConfidence\nVector\nMemory\nPrompt_3\nInstruction: <ins>\nCoT: <cot>\nObservation: <obs>\nMemory: <hist>\nACTION\nLong Term \nMemory\nDataset\nTarget\nMatch\nPrompt\nAction\nWeight\nTask positioning\nCoT generation\nInstruction comprehension\n𝒉𝟑\n𝒄𝟑\nStop\nAppend\nGPT 4\nObs\nFigure 4. The framework of the Multi-Granularity Dynamic Memory (MGDM) model. The CoT feedback module receives task instructions\nand, based on historical observation of corresponding memory, generates a chain of thought and constructs language prompts. The short-\nterm memory module aims to minimize t

In [35]:
rag_retriever.retrieve(" Benchmark for Vision-Language Navigation")


Retrieving documents for query: ' Benchmark for Vision-Language Navigation'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 161.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_06037fae_225',
  'content': 'Towards Long-Horizon Vision-Language Navigation:\nPlatform, Benchmark and Method\nSupplementary Material\n7. Symbol Table\nSymbol Explanation\nIi Observed image from view i\nvi Visual features of view i\noi Fusion visual features of view i\nS Scene representation of current observation\nE Tokenizer\nhi History representation of step i\nHn+1 The memory set of the previous n steps obtained\nat the n + 1th step\nG Large language model\nan Model decision action at step n\nen Expert decision action at step n\nnv The number of viewpoints in the observation.\nIk The indices of the k selected elements\nMst Short term memory\nMlt Long term memory\nC Confidence vector generated from G\nP Pooling function\nTable 5. Symbol Table\n8. Benchmark\n8.1. Trajectory Splitting Algorithm\nWe design a Trajectory Splitting Algorithm 1 for NavGen\nto backward-generate Step-by-Step Navigation Task from\nNavigation Trajectory.\n8.2. Dataset Statistics\nWe conducted stat

## Integrate VectorDB Context  Pipeline with LLM Output

In [46]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

### Initialize the LLM with Gemini 2.5 Flash model and zero temperature for deterministic output

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=1024
)

### Simple RAG function: retrieve context + generate response
def rag_simple(query, retreiver, llm,top_k = 3):

    ### Retrieve relevant documents
    results = retreiver.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant documents found."

    # generate the answer using the retrieved context
    propmt = f"""Use the following context to answer the question concisely.

            Context:
            {context}

            Question: {query}
            Answer:"""
    
    response = llm.invoke([propmt.format(context=context, query=query)])
    return response.content


In [47]:
answer = rag_simple("what is the multi granularity dynamic memory in vln tasks", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'what is the multi granularity dynamic memory in vln tasks'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 121.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The Multi-Granularity Dynamic Memory (MGDM) is a module/model introduced to enhance a model's adaptability and real-world applicability in LH-VLN tasks. It comprises a base model, a Chain-of-Thought (CoT) Feedback module, and Adaptive Memory Integration and Update (AMIU).
